In [2]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, f1_score, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor

In [13]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

In [4]:
# loading the data
fraud_ground_truth = pd.read_csv('fraud_event_ground_truth_may2026.csv')
member_gt = pd.read_csv('member_ground_truth_may2026.csv')
reservations_may = pd.read_csv('reservations_may_2026.csv')
stays_may = pd.read_csv('stays_may_2026.csv')

In [8]:
for name, df in {
    "reservations_may": reservations_may,
    "stays_may": stays_may,
    "fraud_ground_truth": fraud_ground_truth,
    "member_gt": member_gt,
}.items():
    print(f"\n{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("Duplicate confirmation numbers:",
          df["confirmation_number"].duplicated().sum()
          if "confirmation_number" in df.columns else "not applicable")
    print("Most-missing columns:")
    print(df.isna().mean().sort_values(ascending=False).head(8))


reservations_may: 3,200,000 rows, 25 columns
Duplicate confirmation numbers: 0
Most-missing columns:
cancellation_date             0.9949
points_redeemed               0.9375
member_number                 0.4500
member_enrollment_source      0.4500
member_language_preference    0.4500
member_enrollment_date        0.4500
zip                           0.0000
state                         0.0000
dtype: float64

stays_may: 3,200,000 rows, 26 columns
Duplicate confirmation numbers: 0
Most-missing columns:
cancellation_date             0.9949
member_number                 0.4500
member_enrollment_date        0.4500
member_language_preference    0.4500
member_enrollment_source      0.4500
zip                           0.0000
state                         0.0000
city                          0.0000
dtype: float64

fraud_ground_truth: 1,600 rows, 5 columns
Duplicate confirmation numbers: 0
Most-missing columns:
confirmation_number    0.0
member_number          0.0
is_fraud_event         0.0
f

In [ ]:
# # merging the reservations and fraud ground truth dataframes to see how many reservations are fraudulent
# fraud_truth_and_reservations = pd.merge(reservations_may, fraud_ground_truth, how='left', on='confirmation_number')
# fraud_truth_and_reservations.shape[0]

# #filling in the false with 0 instead of na
# fraud_truth_and_reservations['is_fraud_event'] = (
#     fraud_truth_and_reservations['is_fraud_event'].fillna(0).astype(int)
# )

3200000

In [9]:
fraud_truth_and_reservations = pd.merge(
    reservations_may,
    fraud_ground_truth[["confirmation_number", "is_fraud_event"]],
    how="left",
    on="confirmation_number",
    validate="one_to_one",
)

print("Merged shape:", fraud_truth_and_reservations.shape)
print("Reservations with a fraud label:",
      fraud_truth_and_reservations["is_fraud_event"].notna().sum())

Merged shape: (3200000, 26)
Reservations with a fraud label: 1600


In [11]:
fraud_truth_and_reservations["is_fraud_event"] = (
    fraud_truth_and_reservations["is_fraud_event"].fillna(0).astype(int)
)

print(fraud_truth_and_reservations["is_fraud_event"].value_counts())

is_fraud_event
0    3198400
1       1600
Name: count, dtype: int64


In [16]:
fraud_truth_and_reservations["booking_timestamp"] = pd.to_datetime(
    fraud_truth_and_reservations["booking_timestamp"],
    errors="coerce",
    utc=True,
)

booking_date = (
    fraud_truth_and_reservations["booking_timestamp"]
    .dt.tz_convert(None)
    .dt.normalize()
)

check_in_date = pd.to_datetime(
    fraud_truth_and_reservations["check_in_date"],
    errors="coerce",
).dt.normalize()

check_out_date = pd.to_datetime(
    fraud_truth_and_reservations["check_out_date"],
    errors="coerce",
).dt.normalize()

enrollment_date = pd.to_datetime(
    fraud_truth_and_reservations["member_enrollment_date"],
    errors="coerce",
).dt.normalize()

fraud_truth_and_reservations["booking_lead_days"] = (
    check_in_date - booking_date
).dt.days

fraud_truth_and_reservations["stay_nights"] = (
    check_out_date - check_in_date
).dt.days

fraud_truth_and_reservations["booking_hour"] = (
    fraud_truth_and_reservations["booking_timestamp"].dt.hour
)

fraud_truth_and_reservations["booking_day_of_week"] = (
    fraud_truth_and_reservations["booking_timestamp"].dt.dayofweek
)

fraud_truth_and_reservations["member_tenure_days"] = (
    booking_date - enrollment_date
).dt.days

numeric_features = [
    "booking_lead_days",
    "stay_nights",
    "booking_hour",
    "booking_day_of_week",
    "member_tenure_days",
]

categorical_features = [
    "booking_channel",
    "rate_code",
    "site_brand",
]

feature_columns = numeric_features + categorical_features

X = fraud_truth_and_reservations[feature_columns]
y = fraud_truth_and_reservations["is_fraud_event"]

In [17]:
groups = fraud_truth_and_reservations["member_number"].astype("string").fillna("")
nonmember_mask = groups.eq("")

groups.loc[nonmember_mask] = (
    "NONMEMBER_"
    + fraud_truth_and_reservations.loc[
        nonmember_mask, "confirmation_number"
    ].astype(str)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

In [18]:
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train fraud cases:", int(y_train.sum()))
print("Test fraud cases:", int(y_test.sum()))

Train fraud cases: 1269
Test fraud cases: 331


In [19]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=5,
        max_samples=0.5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )),
])

random_forest_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
)

fraud_probabilities = random_forest_model.predict_proba(X_test)[:, 1]
fraud_predictions = (fraud_probabilities >= 0.5).astype(int)

print("Average precision:", average_precision_score(y_test, fraud_probabilities))
print(classification_report(y_test, fraud_predictions, zero_division=0))

tn, fp, fn, tp = confusion_matrix(
    y_test,
    fraud_predictions,
    labels=[0, 1],
).ravel()

false_positive_rate = fp / (fp + tn) if (fp + tn) else 0
print("False-positive rate:", false_positive_rate)
print("True positives:", tp)
print("False positives:", fp)
print("Missed fraud cases:", fn)

Average precision: 0.9891890035106017
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    639492
           1       0.97      1.00      0.99       331

    accuracy                           1.00    639823
   macro avg       0.99      1.00      0.99    639823
weighted avg       1.00      1.00      1.00    639823

False-positive rate: 1.4073670976337468e-05
True positives: 331
False positives: 9
Missed fraud cases: 0
